# Financial phrase sentiment (BERT / FinBERT)

This notebook mirrors the training pipeline in the repository:

- **Data**: `src/training/common.py` — download Financial PhraseBank into `data/`, load as `text` + integer `label` (0=negative, 1=neutral, 2=positive).
- **Classifier fine-tuning**: `src/training/train_classifier.py` — Hugging Face `Trainer`, stratified split, FinBERT label-order handling, optional MLM encoder weights and pseudo-labeled CSV/JSONL.
- **MLM (optional)**: `src/training/train_mlm.py` — continued pre-training on unlabeled `.jsonl` / `.txt`.

## Environment

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (p / "pyproject.toml").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not find pyproject.toml; open the notebook from this repo or set cwd to the repo root.")


REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)],
    cwd=str(REPO_ROOT),
)
print("Editable install complete.")

REPO_ROOT = C:\Users\bryan\Developer\MarketSentimentModel
Editable install complete.


## Imports

In [2]:
import torch

from training import (
    SENTIMENT_ID_TO_STR,
    SENTIMENT_STR_TO_ID,
    default_data_root,
    ensure_finphrasebank,
    load_finphrasebank_dataframe,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("label convention:", SENTIMENT_STR_TO_ID)

device: cpu
label convention: {'negative': 0, 'neutral': 1, 'positive': 2}


## Load Financial PhraseBank

`ensure_finphrasebank()` downloads the same zip as `FINPHRASE_ZIP_URL` in `common.py`, extracts under `data/`, and returns the path to `Sentences_75Agree.txt` by default. Pass `subset="Sentences_50Agree.txt"` (or `66`, `All`) to match other splits.

In [3]:
phrasebank_path = ensure_finphrasebank(data_root=default_data_root())
print("PhraseBank file:", phrasebank_path)

df = load_finphrasebank_dataframe(phrasebank_path)
print("rows:", len(df))
display(df.head())
display(df["label"].map(SENTIMENT_ID_TO_STR).value_counts())

PhraseBank file: C:\Users\bryan\Developer\MarketSentimentModel\data\FinancialPhraseBank-v1.0\FinancialPhraseBank-v1.0\Sentences_75Agree.txt
rows: 3453


,text,label
0,"According to Gran , the company has no plans t...",1
1,With the new production plant the company woul...,2
2,"For the last quarter of 2010 , Componenta 's n...",2
3,"In the third quarter of 2010 , net sales incre...",2
4,Operating profit rose to EUR 13.1 mn from EUR ...,2


label
neutral     2146
positive     887
negative     420
Name: count, dtype: int64

## Fine-tune the classifier (canonical)

The training script is `src/training/train_classifier.py`. It:

- Builds `AutoTokenizer` + `AutoModelForSequenceClassification` from `--base_model`.
- Optionally copies BERT encoder weights from `--mlm_checkpoint` (MLM run).
- Merges optional `--pseudo_data` (CSV/JSONL with `text` and integer `label`).
- Uses stratified `train_test_split` when each class appears at least twice.
- Maps PhraseBank labels to FinBERT order when the checkpoint uses ProsusAI FinBERT’s `id2label`, then **permutes the classifier head back** to PhraseBank order (negative, neutral, positive) before save — same as deployment in the training module docstring.

Run the module below (adjust flags). Output is a Hugging Face model directory (tokenizer + weights).

In [ ]:
import shlex

OUTPUT_DIR = REPO_ROOT / "outputs" / "notebook_classifier"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m",
    "training.train_classifier",
    "--base_model",
    "ProsusAI/finbert",
    "--output_dir",
    str(OUTPUT_DIR),
    "--phrasebank_txt",
    str(phrasebank_path),
    # "--mlm_checkpoint", str(REPO_ROOT / "outputs" / "mlm_bert"),  # optional: encoder from ms-train-mlm
    # "--pseudo_data", str(REPO_ROOT / "data" / "pseudo.jsonl"),   # optional
    "--num_train_epochs",
    "1",  # increase for full training; train_classifier defaults to 3
    "--train_batch_size",
    "16",
    "--eval_batch_size",
    "16",
]
if torch.cuda.is_available():
    cmd.append("--fp16")

print(" ".join(shlex.quote(c) for c in cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT))

'c:\Users\bryan\Developer\MarketSentimentModel\.venv\Scripts\python.exe' -m training.train_classifier --base_model ProsusAI/finbert --output_dir 'C:\Users\bryan\Developer\MarketSentimentModel\outputs\notebook_classifier' --phrasebank_txt 'C:\Users\bryan\Developer\MarketSentimentModel\data\FinancialPhraseBank-v1.0\FinancialPhraseBank-v1.0\Sentences_75Agree.txt' --num_train_epochs 1 --train_batch_size 16 --eval_batch_size 16


## Try the saved checkpoint (optional)

After training, the directory contains `config.json` and weights. A quick smoke test with Hugging Face `pipeline`:

In [ ]:
from transformers import pipeline

cfg = OUTPUT_DIR / "config.json"
if not cfg.is_file():
    print("No saved model yet; run the training cell above.")
else:
    clf = pipeline(
        "text-classification",
        model=str(OUTPUT_DIR),
        tokenizer=str(OUTPUT_DIR),
        truncation=True,
        top_k=None,
        device=0 if torch.cuda.is_available() else -1,
    )
    samples = [
        "The company issued a profit warning.",
        "Operating profit was in line with expectations.",
        "Revenue rose sharply year over year.",
    ]
    for s in samples:
        print(s)
        print(clf(s))
        print()

## Related commands (same code paths)

From a shell in `REPO_ROOT` after editable install:

```bash
ms-train-classifier --base_model ProsusAI/finbert --output_dir outputs/clf_finbert
ms-train-mlm --train_files data/your_corpus.jsonl --output_dir outputs/mlm_bert
```

PhraseBank path is inferred when `--phrasebank_txt` is omitted (download into `data/` via `ensure_finphrasebank`). See `python -m training.train_classifier --help` for all flags (`warmup_ratio`, `val_ratio`, `pseudo_weight`, etc.).